# Stage 4: GARCH, the "stormy weeks follow stormy weeks" formula

Picture a pot of water on the stove. A **big price move** is like stirring hard: it makes waves. The waves **die down slowly** back to the pot's normal calm level. GARCH is a formula that predicts the wave height for tomorrow, the day after, and so on, as the waves settle.

It has three dials:
- **baseline** (`omega`): how choppy this stock normally is
- **reaction** (`alpha`): how strongly a big move yesterday raises today's forecast
- **memory** (`beta`): how slowly the waves die down

"Training" means finding the dial settings that make the stock's real history most plausible. We do it separately for each stock.

Run the cells top to bottom. Kernel: **RiskWise (.venv)**.

In [ ]:
# Cell 1: tools, and the shared helper file we made (pipeline/common.py)
import sys
sys.path.append("../pipeline")           # lets this notebook import our helper file
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from common import load_prices, build_table, yang_zhang, exam_start

STOCKS = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "SBIN"]
prices = {s: load_prices(s) for s in STOCKS}
print({s: len(df) for s, df in prices.items()})

## Step 1: fit GARCH once and look at the dials

We fit on Reliance's first 5 years only (1,250 trading days), just to see what the dials look like. `dist="t"` tells GARCH that real stocks have rare huge days more often than a bell curve expects.

In [ ]:
# Cell 2: fit GARCH once
r = np.log(prices["RELIANCE"]["Close"]).diff().dropna() * 100     # daily moves, in percent
model = arch_model(r.iloc[:1250], mean="Constant", vol="GARCH", p=1, q=1, dist="t")
fit = model.fit(disp="off")
print(fit.params.round(3))

How to read the table:
- `omega`: the baseline pull
- `alpha[1]`: the reaction dial
- `beta[1]`: the memory dial
- `nu`: how heavy the tails are (smaller means more extreme days)
- `mu`: the average daily move (small)

`alpha + beta` is called **persistence**. Close to 1 means storms last a long time.

In [ ]:
# Cell 3: how long do the waves last?
alpha, beta = fit.params["alpha[1]"], fit.params["beta[1]"]
persistence = alpha + beta
print("persistence:", round(persistence, 3))
print("a shock's effect halves every", round(np.log(0.5) / np.log(persistence), 1), "trading days")

In [ ]:
# Cell 4: forecast the next 5 days from this fit
f = fit.forecast(horizon=5).variance.iloc[-1]      # forecast variance for each of the next 5 days
print("daily wiggle forecast, day 1 to day 5:")
print(np.sqrt(f).round(2).to_string())
print("\n5-day wiggle guess:", round(float(np.sqrt(f.mean())), 2), "% a day")

Notice the five numbers drift toward the stock's baseline: that is the waves dying down.

## Step 2: do it honestly, day after day (walk forward)

We need a GARCH guess for **every day** in the history, and each guess may only use the **past**. So:
1. Every 63 trading days (about 3 months) we re-fit the dials, using **only data before that date**.
2. We then use those fixed dials to forecast day by day until the next re-fit.
3. At each day, the forecast uses only that day and earlier days to update the waves.

The function below does exactly that. Read the comments.

In [ ]:
# Cell 5: the walk-forward GARCH function
def garch_guesses(df, refit_every=63, min_train=500, dist="t"):
    r = np.log(df["Close"]).diff().dropna() * 100
    guesses = pd.Series(np.nan, index=r.index)
    for start in range(min_train, len(r), refit_every):
        block_end = min(start + refit_every, len(r))
        # 1) find the dials using ONLY the past: rows before `start`
        spec = arch_model(r.iloc[:start], mean="Constant", vol="GARCH", p=1, q=1, dist=dist)
        dials = spec.fit(disp="off").params
        # 2) freeze those dials, then roll forward day by day through the next block
        rolling = arch_model(r.iloc[:block_end], mean="Constant", vol="GARCH", p=1, q=1, dist=dist).fix(dials)
        forecast = rolling.forecast(horizon=5, start=start - 1, reindex=False).variance
        # 3) average the 5 forecast variances, then square root: the 5-day wiggle guess
        block_guess = np.sqrt(forecast.mean(axis=1)).iloc[: block_end - start]
        guesses.loc[block_guess.index] = block_guess.values
    return guesses

In [ ]:
# Cell 6: run it for all five stocks (about a minute)
guesses = {}
for s in STOCKS:
    guesses[s] = garch_guesses(prices[s])
    print(f"{s:9s} {guesses[s].notna().sum()} daily guesses")

## Proof that GARCH is not peeking

If the guess for a date really uses only the past, then **chopping off all the data after that date must not change it**. The next cell tests exactly that.

In [ ]:
# Cell 7: no-peeking check
df = prices["RELIANCE"]
d = guesses["RELIANCE"].dropna().index[-300]
chopped = df.loc[: d + pd.Timedelta(days=14)]          # keep only about 2 weeks after d
again = garch_guesses(chopped)
print("date:", d.date(), "| full history:", round(guesses["RELIANCE"][d], 5), "| chopped history:", round(again[d], 5))
assert np.isclose(guesses["RELIANCE"][d], again[d]), "the guess changed, so it was using the future!"
print("Same number. No peeking.")

## Step 3: a fair comparison (the scale problem)

GARCH forecasts the wiggle of **closing prices only**. Our target (Yang-Zhang) uses the whole day, so the two sit at slightly different levels. If we compare them directly, GARCH looks worse for a silly reason.

The fix is **one correction number per stock**: the typical ratio between the real wiggle and GARCH's guess, measured **on the study period only**. We also throw away the last 5 study days, because their "real answer" would already reach into the exam.

We also add a tougher yardstick, **Naive-20**: "next week will be as wiggly as the last *month*". GARCH remembers more than one week, so if Naive-20 does as well as GARCH, then GARCH's edge is just "look back further", nothing clever.

In [ ]:
# Cell 8: build, for each stock, the study table, the exam table and the correction number
def prepare(s):
    t = build_table(prices[s]).join(guesses[s].rename("garch_raw"), how="inner")
    t["naive20_guess"] = yang_zhang(prices[s], 20).reindex(t.index)
    t = t.dropna()
    cut = exam_start(t.index)
    study = t[t.index <= cut].iloc[:-5]                 # drop 5 boundary days
    exam = t[t.index > cut].copy()
    scale = (study["actual_next5"] / study["garch_raw"]).median()   # learned from the study period only
    exam["garch_scaled"] = exam["garch_raw"] * scale
    return exam, scale

exams, scales = {}, {}
for s in STOCKS:
    exams[s], scales[s] = prepare(s)
print("correction numbers (real wiggle / GARCH guess):", {s: round(v, 2) for s, v in scales.items()})

## The scoreboard

Same exam as Stage 3 (the most recent 3 years). The score is the **average miss** in points of daily wiggle. Lower is better.

In [ ]:
# Cell 9: the scoreboard
rows = []
for s in STOCKS:
    e = exams[s]
    miss = lambda col: (e["actual_next5"] - e[col]).abs().mean()
    rows.append({
        "stock": s,
        "Naive": miss("naive_guess"),
        "Naive-20 (last month)": miss("naive20_guess"),
        "GARCH raw": miss("garch_raw"),
        "GARCH scaled": miss("garch_scaled"),
    })
board = pd.DataFrame(rows).set_index("stock")
display(board.round(3))

gain = (1 - board["GARCH scaled"] / board["Naive"]) * 100
gain20 = (1 - board["GARCH scaled"] / board["Naive-20 (last month)"]) * 100
print("GARCH scaled beats Naive by (%):")
print(gain.round(1).to_string())
print("\nGARCH scaled beats Naive-20 by (%):  (negative means it LOSES)")
print(gain20.round(1).to_string())

**How to read it.** A positive percentage means GARCH misses by less than that opponent. A negative one means GARCH is *worse*. Be honest with yourself about the second block: Naive-20 is a very cheap trick, so if GARCH does not clearly beat it, GARCH is not adding much.

In [ ]:
# Cell 10: the real wiggle against three guesses, SBIN, last year of the exam
e = exams["SBIN"].iloc[-250:]
e[["actual_next5", "naive_guess", "garch_scaled"]].plot(figsize=(10, 4), title="SBIN: real next-5-day wiggle vs two guesses (% per day)")
plt.ylabel("% per day")
plt.show()

Look at the chart. Does the GARCH line hug the real line more closely than Naive does, especially when a storm starts or ends?

## Your turn

1. In Cell 6, change `garch_guesses(prices[s])` to `garch_guesses(prices[s], refit_every=21)` (re-fit monthly) and re-run Cells 6 to 9. Does the scoreboard change?
2. In Cell 5, change the default `dist="t"` to `dist="normal"` and re-run Cells 6 to 9. Is the "t" setting worth it?
3. In Cell 9, which stock gains the most from GARCH, and which the least? Can you guess why, from what you saw in Cell 3?
4. Explain back: **why must the GARCH dials be fitted only on data from before the day we forecast?**